<div>
    <h1 id = "title"
        style = "background-color: #2788f9;
                 border-radius: 9px;
                 border-left:15px #1e3772 solid;
                 color: #f8f8f8;
                 line-height: 34px;
                 padding: 15px 0px 15px 20px; 
                 font-size: 28px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">Chinese License Plate Recognition [YOLOv8 + CnOCR]
        <a class="anchor-link" id="title" href="chinese-license-plate-recognition-yolov8-cnocr#title">¶</a>
    </h1>
</div>

![Vehicle registration plates of China - Wikipedia](https://upload.wikimedia.org/wikipedia/commons/thumb/3/33/China_Fujian_Sanming_license_plate.jpg/1200px-China_Fujian_Sanming_license_plate.jpg)

<div>
    <h1 id = "preparation"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 border-left:15px #1e3772 solid;
                 color: #1e3772;
                 line-height: 26px;
                 padding: 12px; 
                 font-size: 24px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">1. Preparation
        <a class="anchor-link" id="preparation" href="https://www.kaggle.com/code/harits/chinese-license-plate-recognition-yolov8-cnocr#preparation">¶</a>
    </h1>
</div>

In [ ]:
# ============================================================================
# 导入本项目所需的全部库
# ============================================================================

# --- 忽略警告（保持输出干净） ---
import warnings
warnings.filterwarnings("ignore")

# --- 基础工具库 ---
import os          # 文件/目录操作
import gc          # 垃圾回收
import shutil      # 文件复制等高级文件操作
import time        # 计时
import random      # 随机数
import cv2         # OpenCV：图像读取、画框、颜色转换
import numpy as np # 数值计算
import pandas as pd# 表格数据处理
import glob        # 通配符批量匹配文件路径
from tqdm import tqdm  # 进度条
tqdm.pandas()      # 让 pandas 也能用 tqdm 显示进度
import re          # 正则表达式（清洗识别文字、解析文件名）

# --- 数据可视化 ---
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import Image, display

# --- 深度学习相关 ---
import torch
from numba import cuda  # 用于手动清理 GPU 显存

In [ ]:
# ============================================================================
# CCPD 文件名里的「车牌号编码」需要用到的 3 张字符映射表
# ----------------------------------------------------------------------------
# CCPD 把车牌号编码成下标（见第 3 节的解析），解码时按下标查这 3 个列表：
# ============================================================================

# provinces：车牌第 1 位 = 省份汉字（下标 0~33，含"警""学"等特殊牌照）
provinces = ["皖", "沪", "津", "渝", "冀", "晋", "蒙", "辽", "吉", "黑", "苏", "浙", "京", "闽", "赣", "鲁", "豫", "鄂", "湘", "粤", "桂", "琼", "川", "贵", "云", "藏", "陕", "甘", "青", "宁", "新", "警", "学", "O"]

# alphabets：车牌第 2 位 = 发牌机关代号字母（注意：不含 I 和 O，避免与数字混淆）
alphabets = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W',
             'X', 'Y', 'Z', 'O']

# ads：车牌后 5 位 = 字母 + 数字（共 34 个字符：24 个字母 + 10 个数字）
ads = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X',
       'Y', 'Z', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'O']

# list_sub：6 个子数据集目录名（Base 正常 / FN 远近 / DB 明暗 / Rotate 倾斜 / Weather 天气 / Blur 模糊）
list_sub = ["ccpd_base", "ccpd_fn", "ccpd_db", "ccpd_rotate", "ccpd_weather", "ccpd_blur"]

# 数据集根目录（相对路径，需先解压数据集到该目录）
BASE_PATH = "chinese-city-parking-dataset-eccv"
# 数据集下载地址：https://www.kaggle.com/datasets/binh234/ccpd-preprocess/data
# 代码来源：https://www.kaggle.com/code/harits/chinese-license-plate-recognition-yolov8-cnocr#bbox-and-lpr

<div class="alert alert-block alert-info" style="color: black; font-size: 15px; font-family: Trebuchet MS">
    <b>Sub Datasets</b>:
    <ul>
        <li><b>CCPD-Base</b>: The only common feature of these photos is the inclusion of a license plate</li>
        <li><b>CCPD-FN</b>: The distance from the LP to the shooting location is relatively far or near</li>
        <li><b>CCPD-DB</b>: Illuminations on the LP area are dark, uneven or extremely bright</li>
        <li><b>CCPD-Rotate</b>: Great horizontal tilt degree (20◦
 50◦) and the vertical tilt degree varies from -10◦ to 10</li>
        <li><b>CCPD-Weather</b>: Images taken on a rainy day, snow day or fog day</li>
        <li><b>CCPD-Blur</b>: Blurry largely due to hand jitter while taking pictures</li>
    </ul>
</div>

<div>
    <h1 id = "data-exploration"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 border-left:15px #1e3772 solid;
                 color: #1e3772;
                 line-height: 26px;
                 padding: 12px; 
                 font-size: 24px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">2. Data Exploration
        <a class="anchor-link" id="data-exploration" href="https://www.kaggle.com/code/harits/chinese-license-plate-recognition-yolov8-cnocr#data-exploration">¶</a>
    </h1>
</div>

<div>
    <h2 id = "sample-images"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">2.1. Sample Images
    </h2>
</div>

In [ ]:
# ============================================================================
# 可视化各子数据集（sub-dataset）的样图
# 6 行 = 6 个子数据集；第 1 列写子数据集名，后面 5 列各放一张样图
# ============================================================================

# 创建 6×6 的子图网格（每行 6 列：1 列文字 + 5 列图片）
fig, axs = plt.subplots(6, 6, figsize=(13, 18))

# 逐个处理 6 个子数据集
for i, sub in enumerate(list_sub):
    # 第一列：居中显示子数据集名称
    axs[i, 0].text(0.5, 0.5, sub, ha='center', va='center', fontsize=12)
    axs[i, 0].axis('off')

    # 取该子数据集前 5 个文件
    sub_path = os.path.join(BASE_PATH, sub)
    sub_files = os.listdir(sub_path)[:5]

    # 在剩余 5 列依次显示图片（BGR → RGB 转换，因为 OpenCV 读入是 BGR）
    for j in range(5):
        file_name = os.path.join(sub_path, sub_files[j])
        image = cv2.imread(file_name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        axs[i, j+1].imshow(image)
        axs[i, j+1].axis("off")

# 总标题
plt.suptitle("License Plate Based on Sub-Datasets", x=0.55, y=0.93)

# 显示
plt.show()

<div>
    <h2 id = "image-description"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">2.2. Image Description
    </h2>
</div>

In [ ]:
# ============================================================================
# 解析并打印 CCPD 文件名的各个字段（前 3 张 ccpd_base 图）
# ----------------------------------------------------------------------------
# CCPD 文件名示例：
#   0222389846743-89_86-226,430_509,515-499,524_224,526_225,434_500,432-0_0_5_29_24_25_2-61-24.jpg
#   用 "-" 分成 7 段：
#     [0] 面积比  [1] 倾斜角度  [2] 边界框(左上_右下)  [3] 四个角点  [4] 车牌号编码  [5] 亮度  [6] 模糊度
# ============================================================================
ccpd_base_path = os.path.join(BASE_PATH, "ccpd_base")
ccpd_base_files = os.listdir(ccpd_base_path)

for file in ccpd_base_files[:3]:
    # --- 文件名 ---
    file_name = os.path.join(ccpd_base_path, file)
    file_splitting = file_name.split("/")[-1][:-4]   # 去掉路径和后缀 .jpg
    file_splitting = file_splitting.split("-")       # 按 "-" 拆成 7 段
    print("File Name:", file_name)

    # --- [0] 面积比（车牌区域占整图的比例） ---
    area_ratio = file_splitting[0]
    print("Area Ratio:", area_ratio)

    # --- [1] 倾斜角度（水平_垂直） ---
    tilt_degrees = file_splitting[1]
    hor_tilt_degrees = tilt_degrees.split("_")[0]
    ver_tilt_degrees = tilt_degrees.split("_")[1]
    print("Tilt Degrees")
    print("-  Horizontal:", hor_tilt_degrees)
    print("-  Vertical  :", ver_tilt_degrees)

    # --- [2] 边界框（左上角_右下角） ---
    bounding_box = file_splitting[2]
    left_up_bbox = bounding_box.split("_")[0]
    right_bot_bbox = bounding_box.split("_")[1]
    print("Bounding Box")
    print("-  Left-Up     :", left_up_bbox)
    print("-  Right-Bottom:", right_bot_bbox)

    # --- [3] 四个角点坐标（右下_左下_左上_右上） ---
    vertices = file_splitting[3]
    right_bot_vtc = vertices.split("_")[0]
    left_bot_vtc = vertices.split("_")[1]
    left_up_vtc = vertices.split("_")[2]
    right_up_vtc = vertices.split("_")[3]
    print("Vertices")
    print("-  Right-Bottom:", right_bot_vtc)
    print("-  Left-Bottom :", left_bot_vtc)
    print("-  Left-Up     :", left_up_vtc)
    print("-  Right-Up    :", right_up_vtc)

    # --- [4] 车牌号（下标编码 → 解码成真实车牌） ---
    lcn = file_splitting[4]
    chi_let = provinces[int(lcn.split("_")[0])]      # 第 1 个下标 → 省份
    alp_let = alphabets[int(lcn.split("_")[1])]      # 第 2 个下标 → 字母
    alp_num_let = lcn.split("_")[2:]                 # 后 5 个下标
    alp_num_let = "".join([ads[int(char)] for char in alp_num_let])  # 下标 → 字符
    all_let = chi_let + alp_let + " " + alp_num_let
    print("License Plate Number:", all_let)

    # --- [5] 亮度 ---
    brightness = file_splitting[5]
    print("Brightness:", brightness)

    # --- [6] 模糊度 ---
    blurriness = file_splitting[6]
    print("Blurriness:", blurriness)

    # 显示图片
    img = cv2.imread(file_name)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.show()

<div>
    <h2 id = "bbox-and-lpr"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">2.3. Sample Bounding Box and Exact Vertices
    </h2>
</div>

In [ ]:
# ============================================================================
# 对比可视化「边界框（Bounding Box）」与「精确角点（Exact Vertices）」
# 左列画矩形边界框，右列画四个角点连成的四边形
# ============================================================================

# 创建 3×4 子图（3 行，每行 4 列：边界框、精确角点交替）
fig, axs = plt.subplots(3, 4, figsize=(13, 15))

# 取 ccpd_base 的第 3~8 张图（索引 2:8）
for i, file in enumerate(ccpd_base_files[2:8]):
    # --- 解析文件名 ---
    file_name = os.path.join(ccpd_base_path, file)
    file_splitting = file_name.split("/")[-1][:-4]
    file_splitting = file_splitting.split("-")

    # --- 解码车牌号（用 [4] 段下标映射成真实车牌） ---
    lcn = file_splitting[4]
    chi_let = provinces[int(lcn.split("_")[0])]
    alp_let = alphabets[int(lcn.split("_")[1])]
    alp_num_let = lcn.split("_")[2:]
    alp_num_let = "".join([ads[int(char)] for char in alp_num_let])
    all_let = chi_let + alp_let + alp_num_let

    # --- 解析边界框 [2] 段：左上角_右下角 ---
    # 坐标里用 "," 分隔 x、y，先替换成 "&"，再拆成两个整数
    bounding_box = file_splitting[2]
    lu_bbox = bounding_box.split("_")[0].replace(",", "&")
    x1, y1 = list(map(int, lu_bbox.split("&")))
    rb_bbox = bounding_box.split("_")[1].replace(",", "&")
    x2, y2 = list(map(int, rb_bbox.split("&")))

    # 画矩形边界框
    bbox_img = cv2.imread(file_name)
    bbox_img = cv2.cvtColor(bbox_img, cv2.COLOR_BGR2RGB)
    cv2.rectangle(bbox_img, (x1, y1), (x2, y2), (30, 240, 100), 10)

    # （下方被注释掉的代码：原计划在框上方画车牌号文字，此处保留参考）
#     cv2.rectangle(bbox_img, (x1-50, y1-90), (x2+50, y1), (30, 240, 100), -1, 10)
#     # Add License Plate Number
#     textsize = cv2.getTextSize(all_let, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 4)[0]
#     textX = ((x2-x1+100) - textsize[0]) // 2
#     textY = (90 + textsize[1]) // 2
#     cv2.putText(bbox_img, all_let, (textX+x1-50, textY+y1-90), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,0,0), 4)

    # 左列：显示边界框结果
    axs[i//2, (i*2)%4].imshow(bbox_img)
    axs[i//2, (i*2)%4].set_title("Bounding Box", fontsize=8)
    axs[i//2, (i*2)%4].axis("off")
    axs[i//2, (i*2)%4]

    # --- 解析精确角点 [3] 段，并依次连线 ---
    vertices = file_splitting[3]
    vertices_split = vertices.split("_")

    exver_img = cv2.imread(file_name)
    exver_img = cv2.cvtColor(exver_img, cv2.COLOR_BGR2RGB)
    for j in range(4):
        # 当前角点
        x1, y1 = list(map(int, vertices_split[j].replace(",", "&").split("&")))
        # 下一个角点（%4 保证最后一个点连回第一个点）
        x2, y2 = list(map(int, vertices_split[(j+1)%4].replace(",", "&").split("&")))
        cv2.line(exver_img, (x1, y1), (x2, y2), (30, 240, 100), thickness=10)

    # 右列：显示精确角点结果
    axs[i//2, (i*2+1)%4].imshow(exver_img)
    axs[i//2, (i*2+1)%4].set_title("Exact Vertices", fontsize=8)
    axs[i//2, (i*2+1)%4].axis("off")
    axs[i//2, (i*2+1)%4]

# 总标题
plt.suptitle("Sample Bounding Box and LPR Images", x=0.55, y=0.93)

# 显示
plt.show()

<div>
    <h1 id = "plate-detection"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 border-left:15px #1e3772 solid;
                 color: #1e3772;
                 line-height: 26px;
                 padding: 12px; 
                 font-size: 24px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">3. Plate Detection - YOLOv8
        <a class="anchor-link" id="plate-detection" href="https://www.kaggle.com/code/harits/chinese-license-plate-recognition-yolov8-cnocr#plate-detection">¶</a>
    </h1>
</div>

<div>
    <h2 id = "data-preparation"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">3.1. Data Preparation
    </h2>
</div>

<div>
    <h3 style = "color: #2788f9; font-size: 18px; font-family: Trebuchet MS; font-weight: bold;">Create Metadata</h3>
</div>

In [ ]:
# ============================================================================
# 创建元数据表（Metadata）
# ----------------------------------------------------------------------------
# 目标：CCPD 数据集把「车牌框坐标、车牌号码、亮度、模糊度」等信息全部编码在
#       【文件名】里。这一步就是把文件名解析成一张规整的 DataFrame：
#       每一行 = 一张图片，每一列 = 一个可用字段，方便后面导出 YOLO 标签。
#
# 先记住 CCPD 文件名的结构（用 "-" 分成 7 段）：
#   0222389846743-89_86-226,430_509,515-499,524_224,526_225,434_500,432-0_0_5_29_24_25_2-61-24.jpg
#   └─[0]面积比──┘└[1]倾斜┘└─[2]边界框───┘└────[3]四个角点坐标────┘└──[4]车牌号编码──┘└[5]┘└[6]┘
#                                                                        亮度  模糊度
# ============================================================================


# ---------- 1) 车牌号解码函数 ----------
# 文件名里的 [4] 段形如 "0_0_5_29_24_25_2"，是 7 个用 "_" 分隔的【下标】，
# 必须先经过「下标 → 字符」的映射才能还原成人类可读的车牌号：
#   · 第 1 个数字 → provinces 列表的下标（省份汉字，如 0 → "皖"）
#   · 第 2 个数字 → alphabets 列表的下标（发牌机关字母，如 0 → "A"）
#   · 后 5 个数字 → ads 列表的下标（字母或数字，如 5 → "F"）
# 举例：0_0_5_29_24_25_2 → 皖 + A + F + 1 + 0 + 9 + 8 → "皖AF1098"
def extract_plate_number(plate_number):
    # 第 1 个下标 → 省份汉字
    chi_let = provinces[int(plate_number.split("_")[0])]
    # 第 2 个下标 → 发牌机关字母
    alp_let = alphabets[int(plate_number.split("_")[1])]
    # 剩下 5 个下标（列表切片，返回的还是字符串列表）
    alp_num_let = plate_number.split("_")[2:]
    # 逐个把下标映射成字符，再 join 拼成一个字符串
    alp_num_let = "".join([ads[int(char)] for char in alp_num_let])
    # 省份 + 字母 + 数字，拼出完整车牌
    all_let = chi_let + alp_let + alp_num_let
    return all_let


# ---------- 2) 收集所有图片的路径 ----------
# list_sub 是 6 个子数据集目录（ccpd_base / ccpd_fn / ccpd_db / ...）
# glob 用通配符 "*" 一次性取出某个目录下的全部文件，返回一个列表
# extend：把 glob 的结果「逐个」追加到 list_files（append 会把整个列表当成一个元素）
list_files = []
for sub in list_sub:
    # 例："chinese-city-parking-dataset-eccv/ccpd_base/*"
    sub_path = os.path.join(BASE_PATH, sub, "*")
    list_files.extend(glob.glob(sub_path))

# 直接用一个列表构造 DataFrame，比 pandas 1.x 里已经废弃的 df.append() 循环快得多
df_metadata = pd.DataFrame(list_files, columns=["image_path"])


# ---------- 3) 从路径 / 文件名中解析出各个字段 ----------
# 用 os.path.basename / dirname 取「目录名」和「文件名」，
# 比写死 x.split("/")[5] 更安全：后者要求路径长度固定，换成本地相对路径就会 IndexError
df_metadata["sub"] = df_metadata["image_path"].apply(lambda x: os.path.basename(os.path.dirname(x)))  # 子数据集名
df_metadata["detail"] = df_metadata["image_path"].apply(lambda x: os.path.basename(x))               # 纯文件名

# 取出 [2] 段：边界框，形如 "226,430_509,515"（左上角坐标_右下角坐标）
df_metadata["bbox"] = df_metadata["detail"].apply(lambda x: x.split("-")[2])

# 解析边界框：
#   x.split("_")[0] → 左上角 "226,430"，再按 "," 或 "&" 拆成 x1、y1
#   x.split("_")[1] → 右下角 "509,515"，再按 "," 或 "&" 拆成 x2、y2
#   用 re.split(r"[&,]", ...) 同时兼容 CCPD 的逗号版和其他变体的 & 版
df_metadata["x1_bbox"] = df_metadata["bbox"].apply(lambda x: int(re.split(r"[&,]", x.split("_")[0])[0]))  # 左上角 x
df_metadata["y1_bbox"] = df_metadata["bbox"].apply(lambda x: int(re.split(r"[&,]", x.split("_")[0])[1]))  # 左上角 y
df_metadata["x2_bbox"] = df_metadata["bbox"].apply(lambda x: int(re.split(r"[&,]", x.split("_")[1])[0]))  # 右下角 x
df_metadata["y2_bbox"] = df_metadata["bbox"].apply(lambda x: int(re.split(r"[&,]", x.split("_")[1])[1]))  # 右下角 y

# 取出 [4] 段的车牌号编码，再调用上面写好的函数解码成真实车牌（如 "皖AF1098"）
df_metadata["plate_number"] = df_metadata["detail"].apply(lambda x: x.split("-")[4])
df_metadata["plate_number"] = df_metadata["plate_number"].apply(lambda x: extract_plate_number(x))

# detail 和 bbox 只是解析过程中的中间产物，用完就删掉，让表格保持干净
df_metadata = df_metadata.drop(["detail", "bbox"], axis=1)

# Jupyter 中最后一行表达式会自动显示成表格
df_metadata

<div>
    <h3 style = "color: #2788f9; font-size: 18px; font-family: Trebuchet MS; font-weight: bold;">Create YOLOv8 Bounding Box Format</h3>
</div>

In [ ]:
# ============================================================================
# 把「左上/右下角坐标」转换成 YOLO 需要的「中心点 + 宽高」归一化格式
# ----------------------------------------------------------------------------
# YOLO 标签格式：<class> <x_center> <y_center> <width> <height>
# 所有值都是相对整张图片的【比例】（0~1 之间），不是像素坐标
# CCPD 图片统一是 720×1160（宽×高）
# ============================================================================

# 中心点 x = (x1+x2)/2 / 图宽 720
df_metadata['xmid'] = (df_metadata['x1_bbox'] + df_metadata['x2_bbox']) / (2*720)
# 中心点 y = (y1+y2)/2 / 图高 1160
df_metadata['ymid'] = (df_metadata['y1_bbox'] + df_metadata['y2_bbox']) / (2*1160)

# 框宽 = (x2-x1) / 图宽 720
df_metadata['bb_width'] = (df_metadata['x2_bbox'] - df_metadata['x1_bbox']) / 720
# 框高 = (y2-y1) / 图高 1160
df_metadata['bb_height'] = (df_metadata['y2_bbox'] - df_metadata['y1_bbox']) / 1160

df_metadata

<div>
    <h3 style = "color: #2788f9; font-size: 18px; font-family: Trebuchet MS; font-weight: bold;">Data Splitting</h3>
</div>

In [ ]:
# ============================================================================
# 划分训练集 / 验证集 / 测试集
# ----------------------------------------------------------------------------
# 计划：
#   · Train：从 ccpd_base 随机抽 5000 张
#   · Val / Test：其余 5 个子数据集各抽 100 张，其中 50 张 val + 50 张 test
# 注意：df_metadata 包含全部 27 万多张图片，不能直接整体赋 split，
#       必须按上面的计划抽样后再重组（random_state 固定，结果可复现）
# ============================================================================
parts = []

# 训练集：只从 ccpd_base 抽 5000 张
train_part = df_metadata[df_metadata["sub"] == "ccpd_base"].sample(n=5000, random_state=42)
train_part = train_part.assign(split="train")   # 新增一列 split，值全为 "train"
parts.append(train_part)

# 其余子数据集：各抽 100 张，前 50 张 val、后 50 张 test
for sub in list_sub:
    if sub == "ccpd_base":
        continue
    sub_part = df_metadata[df_metadata["sub"] == sub].sample(n=100, random_state=42)
    sub_part = sub_part.copy()                   # 避免链式赋值的警告
    sub_part["split"] = ["val"] * 50 + ["test"] * 50
    parts.append(sub_part)

# 把所有部分拼回一张表，并重置行索引
df_metadata = pd.concat(parts).reset_index(drop=True)

# 统计每个 split 的样本数
df_metadata["split"].value_counts()

<div>
    <h2 id = "yolo-preparation"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">3.2. YOLOv8 Preparation
    </h2>
</div>

In [ ]:
# 克隆 ultralytics 仓库（YOLOv8 官方实现）
!git clone https://github.com/ultralytics/ultralytics

# 安装 ultralytics 包（提供 YOLO 模型和 yolo 命令行工具）
!pip install ultralytics

In [ ]:
# 在 YOLOv8 目录下创建数据集文件夹结构：
#   datasets/train、datasets/val、datasets/test
# 后面把图片和标签按训练/验证/测试分别放进去
!mkdir "/kaggle/working/ultralytics/datasets"
!mkdir "/kaggle/working/ultralytics/datasets/train"
!mkdir "/kaggle/working/ultralytics/datasets/val"
!mkdir "/kaggle/working/ultralytics/datasets/test"

In [ ]:
# 参考：https://www.kaggle.com/code/aslanahmedov/automatic-number-plate-recognition/notebook
# ============================================================================
# image_and_text_yolo(split)：把某 split（train/val/test）的图片和标签
# 复制到 YOLOv8 数据集目录，并为每张图生成对应的 .txt 标签文件
# ============================================================================
def image_and_text_yolo(split):
    # 取出该 split 的所有样本，重置索引方便遍历
    df = df_metadata[df_metadata["split"]==split].reset_index(drop=True)
    folder_path = os.path.join("/kaggle/working/ultralytics/datasets", split)

    # 只取需要的 5 列：图片路径 + 归一化后的 x、y、w、h
    values = df[['image_path','xmid','ymid','bb_width','bb_height']].values
    for file_name, x, y, w, h in values:
        image_name = os.path.split(file_name)[-1]          # 纯文件名
        txt_name = os.path.splitext(image_name)[0]         # 去掉后缀，作为标签文件名

        dst_image_path = os.path.join(folder_path, image_name)      # 目标图片路径
        dst_label_file = os.path.join(folder_path, txt_name+'.txt') # 目标标签路径

        # 把图片复制到目标目录
        shutil.copy(file_name, dst_image_path)

        # 生成标签文件：0 是类别编号（车牌只有 1 类，所以固定为 0）
        # 格式：0 <x_center> <y_center> <width> <height>
        label_txt = f'0 {x} {y} {w} {h}'
        with open(dst_label_file, mode='w') as f:
            f.write(label_txt)
            f.close()

# 分别处理 train / val / test
image_and_text_yolo("train")
print("Train Dataset Created")
image_and_text_yolo("val")
print("Val Dataset Created")
image_and_text_yolo("test")
print("Test Dataset Created")

In [ ]:
%%writefile /kaggle/working/ultralytics/custom_dataset.yaml
# ============================================================================
# YOLOv8 数据集配置文件（%%writefile 会把下面的内容写入指定文件）
# ----------------------------------------------------------------------------
# train/val/test：三个数据集的路径
# nc：类别数量（本任务只有 1 类：车牌）
# names：类别名称列表
# ============================================================================
train: /kaggle/working/ultralytics/datasets/train
val: /kaggle/working/ultralytics/datasets/val
test: /kaggle/working/ultralytics/datasets/test

nc: 1

names: [
    'license_plate'
]

<div>
    <h2 id = "yolo-training"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">3.3. YOLOv8 Training
    </h2>
</div>

In [ ]:
# 安装 GPUtil：查看 GPU 显存使用情况
!pip install GPUtil
from GPUtil import showUtilization as gpu_usage

# ============================================================================
# free_gpu_cache()：训练前手动清空 GPU 显存缓存，避免显存不足
# ============================================================================
def free_gpu_cache():
    print("Initial GPU Usage")
    gpu_usage()                             # 打印清空前显存使用情况

    torch.cuda.empty_cache()                # 清空 PyTorch 的缓存分配器

    cuda.select_device(0)                   # 选中第 0 块 GPU
    cuda.close()                            # 关闭 CUDA 上下文
    cuda.select_device(0)                   # 重新选中，完成显存重置

    print("GPU Usage after emptying the cache")
    gpu_usage()

free_gpu_cache()

In [ ]:
# 禁用 wandb（实验跟踪平台），避免登录/联网干扰
!wandb disabled

# 开始训练：
#   model=yolov8s.pt  用 YOLOv8 小模型（s = small）的预训练权重
#   data=...          数据集配置文件
#   epochs=25         训练 25 轮
#   batch=32          每批 32 张图
!yolo train model=yolov8s.pt data="./ultralytics/custom_dataset.yaml" epochs=25 verbose=True batch=32

In [ ]:
# 查看训练输出目录下的文件
# 通常包含：weights（best.pt/last.pt 权重）、results.csv（指标）、各种曲线图、confusion_matrix 等
os.listdir("/kaggle/working/runs/detect/train")

<div>
    <h2 id = "loss-metrics-evaluation"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">3.4. Loss and Metrics Evaluation
    </h2>
</div>

In [ ]:
# ============================================================================
# 读取训练过程中的指标记录（每个 epoch 一行）
# ============================================================================
training_results = pd.read_csv("/kaggle/working/runs/detect/train/results.csv")

# 去除列名首尾空格（YOLO 生成的 csv 列名常带空格，会影响后续按列名取值）
list_column = [col.strip() for col in training_results.columns]
training_results.columns = list_column

training_results

In [ ]:
# ============================================================================
# 绘制训练/验证的三种损失（Loss）曲线
#   · box_loss：定位损失（框位置回归误差）
#   · cls_loss：分类损失（是否车牌的判别误差）
#   · dfl_loss：Distributional Focal Loss（边界框分布损失，YOLOv8 特有）
# ============================================================================

# 创建 1 行 3 列的子图
fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["Localization Loss", "Classification Loss", "Distributional Focal Loss"],
)

# 绘图样式配置
class PlotCFG:
    marker_size = 6.5
    line_size = 2
    train_color = "#1e3772"   # 训练曲线颜色
    valid_color = "#2788f9"   # 验证曲线颜色

loss_list = ["box_loss", "cls_loss", "dfl_loss"]
for i, loss in enumerate(loss_list):
    # 训练损失曲线
    fig.add_trace(
        go.Scatter(
            x=np.arange(1, training_results.shape[0]+1), y=training_results["train/{}".format(loss)],
            mode="markers+lines",
            marker=dict(
                color=PlotCFG.train_color, size=PlotCFG.marker_size, line=dict(color="White", width=0.5)
            ),
            line=dict(color=PlotCFG.train_color, width=PlotCFG.line_size),
            name="Training"
        ), row=1, col=i+1
    )
    # 验证损失曲线
    fig.add_trace(
        go.Scatter(
            x=np.arange(1, training_results.shape[0]+1), y=training_results["val/{}".format(loss)],
            mode="markers+lines",
            marker=dict(
                color=PlotCFG.valid_color, size=PlotCFG.marker_size, line=dict(color="White", width=0.5)
            ),
            line=dict(color=PlotCFG.valid_color, width=PlotCFG.line_size),
            name="Validation"
        ), row=1, col=i+1
    )

# 设置坐标轴标题
ticklabels = ["Box Loss", "Cls Loss", "DFL Loss"]
for i, ticklabel in enumerate(ticklabels):
    fig.update_xaxes(title="Epochs", linecolor="Black", ticks="outside", row=1, col=i+1)
    fig.update_yaxes(title=ticklabel, linecolor="Black", ticks="outside", row=1, col=i+1)

# 整体布局设置
fig.update_layout(
    title="Training and Validation Loss", title_x=0.5, font_family="Trebuchet MS",
    width=950, height=350,
    showlegend=False,
    plot_bgcolor="White",
    paper_bgcolor="White"
)

# 显示
fig.show(iframe_connected=True)

In [ ]:
# ============================================================================
# 绘制验证集上的 4 个评估指标曲线
#   · Precision：精确率
#   · Recall：召回率
#   · mAP50：IoU=0.5 时的平均精度均值
#   · mAP50-95：IoU 从 0.5 到 0.95 的平均精度均值
# ============================================================================

# 创建 2×2 子图
subplot_titles = ["Precision", "Recall", "mAP50", "mAP50-95"]
fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=subplot_titles,
)

# 绘图样式配置
class PlotCFG:
    marker_size = 7
    line_size = 2
    train_color = "#1e3772"
    valid_color = "#2788f9"

metrics = ["precision", "recall", "mAP50", "mAP50-95"]
for i, metric in enumerate(metrics):
    # 指标列名形如 "metrics/precision(B)"，所以用 format 拼出完整列名
    fig.add_trace(
        go.Scatter(
            x=np.arange(1, training_results.shape[0]+1), y=training_results["metrics/{}(B)".format(metric)],
            mode="markers+lines",
            marker=dict(
                color=PlotCFG.valid_color, size=PlotCFG.marker_size, line=dict(color="White", width=0.5)
            ),
            line=dict(color=PlotCFG.valid_color, width=PlotCFG.line_size),
            name="Validation"
        ), row=(i//2)+1, col=(i%2)+1
    )

    # 设置坐标轴标题
    fig.update_xaxes(title="Epochs", linecolor="Black", ticks="outside", row=(i//2)+1, col=(i%2)+1)
    fig.update_yaxes(title=subplot_titles[i], linecolor="Black", ticks="outside", row=(i//2)+1, col=(i%2)+1)

# 整体布局设置
fig.update_layout(
    title="Validation Metrics", title_x=0.5, font_family="Trebuchet MS",
    width=950, height=600,
    showlegend=False,
    plot_bgcolor="White",
    paper_bgcolor="White"
)

# 显示
fig.show(iframe_connected=True)

<div>
    <h2 id = "yolov8-testing"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">3.5. YOLOv8 Testing
    </h2>
</div>

<div>
    <h3 style = "color: #2788f9; font-size: 18px; font-family: Trebuchet MS; font-weight: bold;">Model Testing</h3>
</div>

In [ ]:
# 用训练好的 best.pt 权重在测试集（split="test"）上做验证评估
!yolo val model="/kaggle/working/runs/detect/train/weights/best.pt" data="./ultralytics/custom_dataset.yaml" split="test"

In [ ]:
# 查看测试评估输出目录下的文件（含混淆矩阵、P/R/F1/PR 曲线图等）
os.listdir("/kaggle/working/runs/detect/val")

<div>
    <h3 style = "color: #2788f9; font-size: 18px; font-family: Trebuchet MS; font-weight: bold;">Testing Metrics</h3>
</div>

In [ ]:
# ============================================================================
# 显示混淆矩阵（Confusion Matrix）
# ============================================================================
fig = plt.figure(figsize=(10, 9))
cm_img = mpimg.imread("/kaggle/working/runs/detect/val/confusion_matrix.png")
plt.imshow(cm_img)
plt.axis("off")
fig.show()

In [ ]:
# ============================================================================
# 显示 4 张测试曲线图：P_curve（精确率）、R_curve（召回率）、
# F1_curve（F1 分数）、PR_curve（精确率-召回率曲线）
# ============================================================================
fig, axs = plt.subplots(2, 2, figsize=(13, 7.5))
curve_list = ["P_curve", "R_curve", "F1_curve", "PR_curve"]
for i, curve in enumerate(curve_list):
    curve_path = "/kaggle/working/runs/detect/val/{}.png".format(curve)
    curve_img = mpimg.imread(curve_path)
    axs[i//2, i%2].imshow(curve_img)   # i//2 决定行，i%2 决定列
    axs[i//2, i%2].axis('off')

# 总标题
plt.suptitle("Testing Curve", x=0.55, y=0.93)

# 显示
plt.show()

<div>
    <h2 id = "bbox-predictions"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">3.6. Get Bounding Box Predictions
    </h2>
</div>

In [ ]:
# ============================================================================
# 构造结果表 df_results：只保留测试集的图片路径、真实车牌号、真实框坐标
# 后面会往这张表里追加 YOLO 预测框和 CnOCR 识别的车牌号
# ============================================================================
df_results = df_metadata[df_metadata["split"]=="test"].reset_index(drop=True)
df_results = df_results[["image_path", "sub", "plate_number", "x1_bbox", "y1_bbox", "x2_bbox", "y2_bbox"]]
df_results

In [ ]:
# ============================================================================
# 用训练好的 YOLO 模型对测试集图片做车牌检测，收集预测框和置信度
# ============================================================================
from ultralytics import YOLO

# 加载训练好的权重
yolo_model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

# 对整批图片做预测（verbose=False 不打印过程日志）
bbox_results = yolo_model.predict(list(df_results["image_path"]), verbose=False)

all_box_list = []   # 每张图的预测框列表（列表的列表）
all_conf_list = []  # 每张图的置信度列表（列表的列表）

for bbox_result in bbox_results:
    boxes = bbox_result.boxes     # 当前图片的所有检测框
    box_list = []
    conf_list = []
    for box in boxes:
        box_data = box.data[0][:4]          # 取前 4 个值：x1, y1, x2, y2
        box_data = [int(x) for x in box_data]  # 转成整数像素坐标
        conf = round(float(box.conf), 2)    # 置信度保留 2 位小数
        if(conf>=0.5):                      # 只保留置信度 ≥ 0.5 的框
            box_list.append(box_data)
            conf_list.append(conf)

    all_box_list.append(box_list)
    all_conf_list.append(conf_list)

In [ ]:
# 把预测框和置信度写入结果表
df_results["pred_bbox"] = all_box_list     # 预测的车牌框坐标
df_results["confidence"] = all_conf_list   # 对应的置信度
df_results

<div>
    <h2 id = "show-test-images-detection"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">3.7. Show Test Images after Detection
    </h2>
</div>

In [ ]:
# 在 Jupyter 内联显示图片
%matplotlib inline
from ultralytics.yolo.utils.plotting import Annotator

# ============================================================================
# show_plate_detection(idx)：可视化第 idx 张测试图的车牌检测结果
#   1) 在原图上画出预测框和标签
#   2) 同时把裁剪出来的车牌区域单独显示
# ============================================================================
def show_plate_detection(idx):
    # 读入原图并转 RGB
    ori_path = df_results["image_path"][idx]
    print(ori_path)
    ori_image = cv2.imread(ori_path)
    ori_image = cv2.cvtColor(ori_image, cv2.COLOR_BGR2RGB)

    # 用 Annotator 在图上画框和文字
    annotator = Annotator(ori_image)
    boxes = df_results["pred_bbox"][idx]
    confs = df_results["confidence"][idx]
    for i, box in enumerate(boxes):
        # 画框 + 标签（类别名 + 置信度）
        annotator.box_label(box=box, label="license_plate " + (str(confs[i])), color=(30, 240, 100))

        # 裁剪出车牌区域并单独显示
        x1, y1, x2, y2 = box
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        roi = ori_image[y1:y2, x1:x2]      # 注意切片是 [y1:y2, x1:x2]
        plt.imshow(roi)
        plt.show()

    # 显示画好框的完整图片
    frame = annotator.result()
    plt.imshow(frame)
    plt.show()

# 挑 5 张测试图展示检测结果
idxs = [0, 50, 100, 150, 200]
for idx in idxs:
    show_plate_detection(idx)

<div>
    <h1 id = "plate-recognition"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 border-left:15px #1e3772 solid;
                 color: #1e3772;
                 line-height: 26px;
                 padding: 12px; 
                 font-size: 24px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">4. Plate Recognition - CnOCR
        <a class="anchor-link" id="plate-recognition" href="https://www.kaggle.com/code/harits/chinese-license-plate-recognition-yolov8-cnocr#plate-recognition">¶</a>
    </h1>
</div>

<div>
    <h2 id = "text-predictions"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">4.1. Get Text Predictions
    </h2>
</div>

In [ ]:
# 安装 CnOCR：轻量级中英文 OCR 库，用于识别裁剪出的车牌文字
!pip install cnocr

In [ ]:
from cnocr import CnOcr

# ============================================================================
# get_text_cnocr(idx)：对第 idx 张测试图做「车牌文字识别」
# ----------------------------------------------------------------------------
# 流程：读图 → 取出 YOLO 预测的车牌框 → 逐个裁剪 → CnOCR 识别 → 清洗结果
# 返回：该图所有车牌框识别出的文字列表（每块车牌一个字符串）
# ============================================================================
def get_text_cnocr(idx):
    # 1) 读入原图，并统一转成 RGB（OpenCV 默认读成 BGR，CnOCR 需要 RGB）
    ori_path = df_results["image_path"][idx]
    ori_image = cv2.imread(ori_path)
    ori_image = cv2.cvtColor(ori_image, cv2.COLOR_BGR2RGB)

    # 2) 创建 CnOCR 模型
    #    注意：模型在函数内创建，每调用一次都会重新加载，适合演示；批量时可移到外面只加载一次
    cnocr_model = CnOcr()

    # 3) 取出该图所有已预测的车牌框，逐个识别
    text_list = []
    bboxs = df_results["pred_bbox"][idx]
    for bbox in bboxs:
        # Crop Image with Bounding Box
        # bbox 是 [x1, y1, x2, y2]（左上角、右下角）
        # 注意 NumPy 切片顺序是 [行, 列] = [y, x]，所以先 y 后 x
        crop_image = ori_image[bbox[1]:bbox[3], bbox[0]:bbox[2]]

        # Extract Plate Number
        # 调用 CnOCR 识别裁剪出来的车牌图片
        text_output = cnocr_model.ocr(crop_image)
        if(len(text_output)>0):
            # ocr() 返回列表，每个元素是一个 dict（含 text/score 等），取第 1 个结果里的文字
            text = text_output[0]["text"]

            # Clean Text —— 清洗识别结果：
            #   1) 去掉空格和常见分隔符（·、:、-）
            #   2) 纠正易混淆字符：O→0、I→1
            #   3) 全部转成大写
            del_punc_list = [" ", "·", ":", "-"]   # 备用字符表（此处未实际使用，仅作说明）
            clean_text = re.sub(r"[ ·:-]", "", text)
            clean_text = clean_text.replace("O", "0")
            clean_text = clean_text.replace("I", "1")
            clean_text = clean_text.upper()
            text_list.append(clean_text)

    return text_list

In [ ]:
# 批量识别测试集前 250 张图片的车牌文字
idxs = list(np.arange(0, 250))

# tqdm 包裹列表推导式，边跑边显示进度条
text_results = tqdm([get_text_cnocr(idx) for idx in idxs])

# 每张图可能识别出多个车牌框，这里只取第一个结果；没有结果则用空字符串占位
text_results = [text[0] if len(text)>0 else "" for text in text_results]

# 把识别结果写回表格
df_results["pred_plate_number"] = text_results
df_results

<div>
    <h2 id = "show-test-images-detection"
        style = "background-color: #9dd6fb;
                 border-radius: 15px;
                 color: #1e3772;
                 line-height: 20px;
                 padding: 11px; 
                 font-size: 20px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">4.2. Show Test Images after Text Recognition
    </h2>
</div>

In [ ]:
# 在 Jupyter 内联显示图片
%matplotlib inline

# 挑 5 张测试图：先可视化车牌检测框，再打印识别出的文字
idxs = [0, 50, 100, 150, 200]
for idx in idxs:
    show_plate_detection(idx)      # 显示车牌框 + 裁剪出的车牌区域
    text_ocr = get_text_cnocr(idx) # 对同一张图做文字识别
    print(text_ocr)

<div>
    <h2 id = "title"
        style = "background-color: #2788f9;
                 border-radius: 9px;
                 color: #f8f8f8;
                 line-height: 50px;
                 padding: 15px 0px 15px 20px; 
                 text-align: center;
                 font-size: 28px;
                 font-weight: bold;
                 font-family: Trebuchet MS;">To be continued
    </h2>
</div>

<div class="alert alert-block alert-info" style="color: black; font-size: 15px; font-family: Trebuchet MS">
    <b>References</b>:
    <ul>
        <li>CCPD (Chinese City Parking Dataset, ECCV): <a href="https://github.com/detectRecog/CCPD"><u>https://github.com/detectRecog/CCPD</u></a></li>
        <li>Towards End-to-End License Plate Detection and Recognition: A Large Dataset and Baseline; Xu, Zhenbo and Yang, Wei and Meng, Ajin and Lu, Nanxue and Huang, Huan; Proceedings of the European Conference on Computer Vision (ECCV); 255--271; 2018</li>
        <li>Automatic Number Plate Recognition Notebook - Aslan Ahmedov: <a href="https://www.kaggle.com/code/aslanahmedov/automatic-number-plate-recognition"><u>https://www.kaggle.com/code/aslanahmedov/automatic-number-plate-recognition</u></a></li>
    </ul>
    <b>Improvements</b>:
    <ul>
        <li>Improve image quality to get better text extraction results after cropping from model detection.</li>
        <li>It can use a model that performs both plate detection and recognition simultaneously such as RPNet from main paper sources.</li>
    </ul>
</div>